In [18]:
from pydantic import BaseModel, Field
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langsmith import traceable, get_current_run_tree, Client
from operator import add
from typing import Any, Annotated, Dict, List
import yaml
from jinja2 import Template

from langchain_core.messages import BaseMessage, AIMessage, ToolMessage, HumanMessage, SystemMessage
from langchain_core.tools import BaseTool
from langchain_core.prompts import BasePromptTemplate
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.prompts import StringPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import convert_to_openai_messages, convert_to_messages
from langchain_protocol import Literal

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document,Prefetch, FusionQuery
from qdrant_client import models

import instructor

import pandas as pd
import openai
import fastembed

from jinja2 import Template
from typing import List, Dict, Any, Optional, Union
from IPython.display import Image, display
from operator import add
from openai import OpenAI
import tiktoken

import random
import ast
import inspect
import instructor
import json
import os
import importlib
import utils
from dotenv import load_dotenv
import numpy as np
load_dotenv()
importlib.reload(utils)

from utils import format_ai_message, parse_function_definition, get_type_from_annotation, parse_docstring_params, get_tool_descriptions

In [19]:
ls_client = Client()
ls_prompt = ls_client.pull_prompt("retrieval_generation_prompt")
ls_template = ls_prompt.messages[0].prompt.template

preprocessed_context = "- a \n - b"
question = "What is a?"

def build_prompt_with_jinja(preprocessed_context, question):
    jinja_template = """You are a helpful shopping assistant for answering questions about products in stock.
      You will be given a question and a list of context

      Instructions:
      - You need to answer the question based on the provided context only
      - Never use word context and refer to it as the available products
      - As an output you need to provide:

      * The answer to the question based on the provided context
      * The list of the IDs of the chuns that were used to answer the question.
      only return the ones that are used in the answer.
      * Short description (1-2 sentences) of the item based on the description provided in the context

      - The short description should have the name of the item.
      - The answer to the question should contain detailed information about the product and returned with
      detailed specification in bullet points.

      Context:
        {{preprocessed_context}}
      Question: 
        {{question}}
    """

    template = Template(jinja_template)
    rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)
    return rendered_template

def prompt_template_config(yaml_file, prompt_key):
    with open(yaml_file, 'r') as file:
        config = yaml.safe_load(file)

    prompt_entry = config['prompts'][prompt_key]
    template_content = prompt_entry['template'] if isinstance(prompt_entry, dict) else prompt_entry

    template = Template(template_content)

    return template


def prompt_template_registry(prompt_name):
    template_content = ls_client.pull_prompt(prompt_name).messages[0].prompt.template
    template = Template(template_content)
    return template


print(prompt_template_registry("retrieval_generation_prompt").render(preprocessed_context=preprocessed_context, question=question))

You are a helpful shopping assistant for answering questions about products in stock.
      You will be given a question and a list of context
      Instructions:
      - You need to answer the question based on the provided context only
      - Never use word context and refer to it as the available products
      - As an output you need to provide:
      * The answer to the question based on the provided context
      * The list of the IDs of the chuns that were used to answer the question.
      only return the ones that are used in the answer.
      * Short description (1-2 sentences) of the item based on the description provided in the context
      - The short description should have the name of the item.
      - The answer to the question should contain detailed information about the product and returned with
      detailed specification in bullet points.
      Context:
- a 
 - b
      Question: 
What is a?


In [20]:

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")

qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key,
)

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_94931/3853642596.py:18: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(


Retrieve all item IDs from Amazon Items Qdrant Collection

In [21]:
import openai

# 1. Generate the embedding vector for your search term
response = openai.embeddings.create(
    input="electronics",
    model="text-embedding-3-small"
)
search_vector = response.data[0].embedding

# 2. Pass the numerical search_vector to query_points
payload = qdrant_client.query_points(
    collection_name="Amazon_Electronics_Products",
    query=search_vector,               # Pass the list of floats here
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False                 # Note: in query_points it is with_vectors, not with_vector
)


In [22]:
payload.points

[ScoredPoint(id=1629, version=13, score=0.34347615, payload={'parent_asin': 'B003YD34PI'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1353, version=11, score=0.34145623, payload={'parent_asin': 'B07F8C1PMQ'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1867, version=15, score=0.333336, payload={'parent_asin': 'B01LX1Z17Z'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=322, version=3, score=0.33064395, payload={'parent_asin': 'B00HSX9ZB2'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=880, version=7, score=0.32894048, payload={'parent_asin': 'B00ALKJODS'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=746, version=6, score=0.3276618, payload={'parent_asin': 'B078TQMXKW'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1675, version=14, score=0.3255921, payload={'parent_asin': 'B000GAU7AM'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1762, version=14, sco

In [23]:
len(payload.points)

1000

In [24]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]

In [25]:
parent_asin_list

['B003YD34PI',
 'B07F8C1PMQ',
 'B01LX1Z17Z',
 'B00HSX9ZB2',
 'B00ALKJODS',
 'B078TQMXKW',
 'B000GAU7AM',
 'B011NAANHU',
 'B00SLKQRJO',
 'B01NBXUPMX',
 'B0B2RKFTPH',
 'B001DRULBM',
 'B09YLX1834',
 'B0038L1XYU',
 'B000E8VCKU',
 'B0BT1TM3S2',
 'B0829ZQ97G',
 'B00330U8UQ',
 'B09CL2VNPJ',
 'B00DKFJA2G',
 'B07RY1X43Z',
 'B07L2W17V6',
 'B001963Y9I',
 'B0B3SMDK54',
 'B083C4BLVQ',
 'B000OMQXF0',
 'B071NLL6Z3',
 'B0075X3F7K',
 'B007884QFW',
 'B00JVIPOC6',
 'B01CZFUOD0',
 'B088XCZM3B',
 'B00IEFS0A0',
 'B07ZWYLYFR',
 'B07CQVWJH1',
 'B00AB0YYJM',
 'B00OT7HSBO',
 'B07FN4CZTR',
 'B0B5Y4FVYY',
 'B00AZMP9BS',
 'B001JVPFV8',
 'B087RMRLNK',
 'B01M6CLLFN',
 'B00R17YGL4',
 'B00XHY9YTE',
 'B00HQ9DJS8',
 'B07HYG9FPT',
 'B01GYJJH5Y',
 'B00L7O2AGA',
 'B0983DRDTR',
 'B0BTQW3CBP',
 'B000ZSV4X4',
 'B083PQ4WDK',
 'B001RS5ASG',
 'B08T8BSHQ9',
 'B01443P08K',
 'B07GZNGX4F',
 'B075P7WPJT',
 'B07H2Z829Y',
 'B01DVL0GQ2',
 'B07Y1YBYV3',
 'B000Q6M5CY',
 'B08LMVFTL5',
 'B07QZYRRBR',
 'B01161BYG0',
 'B002RYV1T6',
 'B010ELNK

Load Amazon Electronics Dataset

In [26]:
df_reviews = pd.read_json("../../data/Review_Data.jsonl", lines = True)

In [27]:
df_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True
1,1,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True
2,5,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True
3,5,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,2010-11-20 18:41:35.000,18,True
4,5,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,2023-02-17 02:39:41.238,0,True
...,...,...,...,...,...,...,...,...,...,...
56351,5,Five Stars,"item as described,fast shipping",[],B00YBCKTM2,B00YBCKTM2,AG4LJHP2T2IRDCADZRRSDROMH4AQ,2017-06-28 20:56:13.180,0,True
56352,5,Five Stars,"item as described,fast shipping",[],B01LWRUX6B,B01LWRUX6B,AG4LJHP2T2IRDCADZRRSDROMH4AQ,2017-06-28 20:51:52.910,0,True
56353,5,Five Stars,"item as described,fast shipping",[],B00N2VIALK,B00O1RTQJE,AG4LJHP2T2IRDCADZRRSDROMH4AQ,2017-06-28 20:50:36.926,0,True
56354,5,HP 24GO1623.8 ALL IN 1 COMPUTER,LOVE THE MONITOR ON THIS COMPUTER.PICTURE IS C...,[],B01G2AJ5MK,B01G2AJ5MK,AG4LJHP2T2IRDCADZRRSDROMH4AQ,2017-02-04 23:51:51.000,3,True


In [28]:
len(df_reviews)

56356

In [29]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True
1,1,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True
2,5,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True
3,5,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,2010-11-20 18:41:35.000,18,True
4,5,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,2023-02-17 02:39:41.238,0,True


In [30]:
df_reviews_sample = df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]

In [31]:
len(df_reviews_sample)

64

Define functions to preprocess review data

In [32]:
def preprocess_reviews_data(row):
    return f"{row["title"]} {row["text"]}"

In [33]:
encoding = tiktoken.encoding_for_model("text-embedding-3-small")

In [34]:
encoding.encode("Can I get earphones?")

[6854, 358, 636, 2487, 17144, 30]

In [35]:
import tiktoken
def token_count(row, model="text-embedding-3-small"):
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(row["preprocessed_data"]))

In [37]:
df_reviews_sample["preprocessed_data"] = df_reviews_sample.apply(preprocess_reviews_data, axis=1)

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_94931/1882656165.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reviews_sample["preprocessed_data"] = df_reviews_sample.apply(preprocess_reviews_data, axis=1)


In [38]:
df_reviews_sample["preprocessed_data_token_count"] = df_reviews_sample.apply(token_count, axis=1)

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_94931/1530805122.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reviews_sample["preprocessed_data_token_count"] = df_reviews_sample.apply(token_count, axis=1)


In [39]:
df_reviews_sample.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,preprocessed_data_token_count
3213,5,Absolutely Perfect! Case Holds All Basic Versa...,This APROCA HARD STORAGE TRAVEL CASE FOR DREME...,[],B07ZZ595TG,B07ZZ595TG,AEYGPUCRKH7G4VM22FM3VAKSQ23Q,2020-06-24 18:08:34.606,5,True,Absolutely Perfect! Case Holds All Basic Versa...,259
4396,5,"As Described, Quality Product, Fast Delivery. ...","As Described, Quality Product, Fast Delivery. ...",[],B087K9L3R5,B08B1WQY3D,AFDSTZALRDM64PFS64CMKVIZUN6Q,2021-05-16 19:17:03.190,0,True,"As Described, Quality Product, Fast Delivery. ...",26
5046,5,Good Quality for the Price,The sound quality on these headphones is surpr...,[],B07D4GXQTM,B07WF9SLQP,AFQQQ5LGNSQUEBGDCYBAZZE5T3DA,2020-03-11 22:02:39.397,2,False,Good Quality for the Price The sound quality o...,94
6190,2,Short life span,Worked for about 4 months and then just stoppe...,[],B07VXWXLGZ,B09ZTC6LGP,AEQ7Y56NEUZTFG2QJYQ4T32EIUZA,2020-07-12 17:56:48.911,0,True,Short life span Worked for about 4 months and ...,32
6332,3,Three Stars,It is a decent cable,[],B00BFY8E1C,B09G3CL3Y3,AEDQPYY76IMRKDA2HGWZ7STA6V2A,2015-10-04 00:25:07.000,0,True,Three Stars It is a decent cable,7


In [40]:
len(df_reviews_sample)

64

In [41]:
df_reviews_sample = df_reviews_sample[df_reviews_sample["preprocessed_data_token_count"] < 8192]

In [42]:
len(df_reviews_sample)

64

In [43]:
total_tokens = df_reviews_sample["preprocessed_data_token_count"].sum()

In [44]:
total_tokens

np.int64(4851)

Create a new Qdrant collection for reviews data

In [45]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-reviews",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

True

In [46]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-reviews",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

Embedding Functions

In [47]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding

In [48]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings = []
    counter = 1

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend(embedding.embedding for embedding in response.data)
        print(f"Processed {counter + batch_size} of {len(text_list)}")
        counter += 1

    return all_embeddings

EMbed the text and add additional fields to the payload of each vector for reviews

In [49]:
data_to_embed_reviews = df_reviews_sample[["preprocessed_data", "parent_asin"]].to_dict(orient="records")

In [50]:
data_to_embed_reviews

[{'preprocessed_data': 'Absolutely Perfect! Case Holds All Basic Versa Parts--And More This APROCA HARD STORAGE TRAVEL CASE FOR DREMEL VERSA CLEANING TOOL PC10 is absolutely perfect!  The Dremel Versa fits in one half of the case, along with the bristle brush and splash guard (nested).  The other half of the case holds the cleaning disc holder, at least 9 assorted cleaning discs (white, blue, brown), a long brush (an extra accessory, available on Amazon), and the Versa charger and charging cable.<br /><br />The hard case has a good zipper closure that zips from either side.  It also has a ribbon carry handle.  When closed, the Versa tool and all of its accessories are contained in a single 9”W x 5”H x 3.5”D case.<br /><br />Looking at the product page photo, I wasn’t sure whether the bristle brush would fit in this case, but it does--and the case even has room for a different brush and a bunch of cleaning discs.  I’m really glad that I ordered this, because it is so much handier to use

In [51]:
len(data_to_embed_reviews)

64

In [53]:
text_to_embed_reviews = [data["preprocessed_data"] for data in data_to_embed_reviews]

In [54]:
text_to_embed_reviews

['Absolutely Perfect! Case Holds All Basic Versa Parts--And More This APROCA HARD STORAGE TRAVEL CASE FOR DREMEL VERSA CLEANING TOOL PC10 is absolutely perfect!  The Dremel Versa fits in one half of the case, along with the bristle brush and splash guard (nested).  The other half of the case holds the cleaning disc holder, at least 9 assorted cleaning discs (white, blue, brown), a long brush (an extra accessory, available on Amazon), and the Versa charger and charging cable.<br /><br />The hard case has a good zipper closure that zips from either side.  It also has a ribbon carry handle.  When closed, the Versa tool and all of its accessories are contained in a single 9”W x 5”H x 3.5”D case.<br /><br />Looking at the product page photo, I wasn’t sure whether the bristle brush would fit in this case, but it does--and the case even has room for a different brush and a bunch of cleaning discs.  I’m really glad that I ordered this, because it is so much handier to use than a ZipLoc storage

In [55]:
embedding_reviews = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

In [56]:
embedding_reviews

[[0.01203155517578125,
  0.0169219970703125,
  -0.00865936279296875,
  0.00504302978515625,
  0.003673553466796875,
  -0.006256103515625,
  0.0183563232421875,
  0.0272064208984375,
  0.0076446533203125,
  0.03369140625,
  0.053558349609375,
  -0.023651123046875,
  0.01073455810546875,
  -0.0240478515625,
  0.046173095703125,
  0.0723876953125,
  -0.007038116455078125,
  -0.00817108154296875,
  0.004047393798828125,
  0.0158538818359375,
  0.0408935546875,
  0.051910400390625,
  0.06158447265625,
  -0.0186309814453125,
  0.01507568359375,
  -0.043182373046875,
  0.00433349609375,
  0.0088958740234375,
  0.060760498046875,
  0.003108978271484375,
  -0.021728515625,
  -0.01216888427734375,
  -0.01535797119140625,
  -0.019927978515625,
  0.005268096923828125,
  0.033477783203125,
  0.0010442733764648438,
  -0.059906005859375,
  -0.01450347900390625,
  0.042510986328125,
  -0.0160369873046875,
  0.055999755859375,
  0.01318359375,
  0.01517486572265625,
  0.0306854248046875,
  -0.016265869

In [58]:
len(embedding_reviews)

64

In [63]:
pointstructs = []
i = 1
for embedding, data in zip(embedding_reviews, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload={
                "text": data["preprocessed_data"],
                "parent_asin": data["parent_asin"]
            }
        )
    )
    i += 1


In [64]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-items-collection-reviews",
        wait=True,
        points=batch
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

Processed 100 of 64


A function to run search against reviews on a prefiltered set of product IDs

In [ ]:
def retrieve_prefiltered_reviews_data(query, parent_asin, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-reviews",
        prefetch=[
            Prefetch(
                query=query_embedding,
                filter=models.Filter(
                    must=[
                        models.FieldCondition(
                            key="parent_asin",
                            match=models.MatchAny(
                                any=parent_asin
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    return results

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (2723613483.py, line 8)